# E-commerce Orders - Exploratory Analysis

This notebook walks through the same pipeline as `python -m src.run_analysis`,
one business question at a time. Every number comes from `src/`, so the
notebook and the CLI report can never drift apart.

In [ ]:
import pandas as pd

from src import analysis, data_cleaning, data_loader, feature_engineering

pd.set_option("display.float_format", lambda v: f"{v:,.2f}")

## 1. Load and clean the raw table
The raw CSV is read entirely as text, then cleaned with the 21-step checklist.
The audit log records the reasoning for every single action.

In [ ]:
raw = data_loader.load_and_validate()
clean, log, metrics = data_cleaning.clean_orders(raw)
print(f"{raw.shape} -> {clean.shape} | quality {metrics['score_before']} -> {metrics['score_after']}")
pd.DataFrame(log)

## 2. Derive analysis fields

In [ ]:
df = feature_engineering.add_features(clean)
df[["order_id", "product", "revenue", "status", "order_year_month"]].head()

## 3. Which products drive the business?

In [ ]:
by_product = analysis.revenue_by_product(df)
ax = by_product.set_index("product")["revenue"].plot.barh(figsize=(8, 5), title="Revenue by product")
ax.invert_yaxis()
by_product

## 4. Is revenue seasonal?

In [ ]:
trend = analysis.revenue_trend(df)
trend.plot(x="order_year_month", y="revenue", marker="o", figsize=(9, 4), title="Monthly revenue")

## 5. Where is the money coming from?

In [ ]:
analysis.revenue_by_country(df)

In [ ]:
analysis.revenue_by_payment(df)

## 6. How much money is lost to cancellations and refunds?

In [ ]:
analysis.cancellation_impact(df)

## 7. Machine learning

Two algorithms: an order-outcome classifier and KMeans RFM segmentation.

In [ ]:
from src import ml

ml_result = ml.train_order_outcome_model(df)
ml_result["cv_table"]

In [ ]:
segments = ml.segment_customers(df)
segments["summary"]

## 8. Headline KPIs

Run `python -m src.run_analysis` to regenerate every chart and the
self-contained HTML dashboard in `reports/dashboard.html`.

In [ ]:
analysis.overview_metrics(df)